# Day 07 - Window Functions

**Topic:** Window Functions  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกใช้ Window Functions เช่น `row_number`, `rank`, `dense_rank`, `lag`, `lead` เพื่อแก้ปัญหา latest record, deduplication และ event sequence analysis ใน PySpark

วันนี้เราจะเรียน Window Functions ใน PySpark ผ่านโจทย์ที่เจอได้บ่อยในงาน Data Engineering เช่น เลือก record ล่าสุดต่อ customer, จัดอันดับ transaction ภายในกลุ่ม, เปรียบเทียบค่าปัจจุบันกับค่าก่อนหน้า และคำนวณ running total โดยยังคง detail-level rows ไว้


## Goal of the day

1. อธิบายได้ว่า **Window Function** ต่างจาก `groupBy().agg()` อย่างไร
2. ใช้ `Window.partitionBy()` และ `Window.orderBy()` เพื่อกำหนดกรอบการคำนวณได้
3. ใช้ `row_number()` เพื่อเลือก latest record ต่อ business key ได้
4. ใช้ `rank()` และ `dense_rank()` เพื่อจัดอันดับ records ภายใน group ได้
5. ใช้ `lag()` และ `lead()` เพื่อเปรียบเทียบ current row กับ previous / next row ได้
6. เริ่มระวังเรื่อง tie-breaker, ordering และ window ที่กว้างเกินไปในงาน production
   - tie-breaker คือ “เงื่อนไขสำรองที่ใช้ตัดสินลำดับ” ตอนที่ค่าหลักที่ใช้ `orderBy` เท่ากัน เช่น `updated_at` เท่ากัน แต่ยังต้องเลือก record ล่าสุดเพียง 1 record


## Why it matters in production

ใน production pipeline Window Functions สำคัญมาก เพราะหลายโจทย์ไม่ได้ต้องการแค่ summary ระดับ group แต่ต้องการคำนวณบน rows ภายใน group โดยยังเก็บ row-level detail ไว้ เช่น:

- เลือก customer record ล่าสุดจาก source history
- deduplicate records ด้วย business key และ timestamp
- เตรียมข้อมูลสำหรับ SCD / history logic
- หา previous status, previous amount หรือ previous event
- ทำ running total หรือ sequence-based validation
- ตรวจสอบ event order เช่น transaction ครั้งก่อนหน้าและครั้งถัดไป

Production takeaway:

> ก่อนใช้ Window Function ต้องรู้ว่า `partition` คือกลุ่มอะไร และ `orderBy` ใช้ logic อะไรในการจัดลำดับ เพื่อให้ผลลัพธ์ deterministic หรือรันซ้ำแล้วได้ผลลัพธ์เดิมเสมอ


## Key concepts

| Concept | Meaning |
|---|---|
| Window Function | Function ที่คำนวณค่าแบบอิงกลุ่มข้อมูล แต่ยังคงจำนวน rows เดิมไว้ ไม่เหมือน `groupBy` ที่ยุบ rows |
| Window Specification | กติกาของ window เช่น แบ่งกลุ่มด้วย `partitionBy` และจัดลำดับในกลุ่มด้วย `orderBy` |
| `partitionBy` | แบ่งกลุ่ม rows ที่ window function จะคำนวณแยกกัน เช่น ต่อ `customer_id` |
| `orderBy` | กำหนดลำดับภายใน partition เช่น `updated_at` ล่าสุดก่อน |
| `row_number` | ให้เลขลำดับไม่ซ้ำในแต่ละ partition ใช้บ่อยกับ latest-record deduplication |
| `rank` | จัดอันดับแบบมี gap เมื่อมี tie หรือค่าที่ใช้เรียงลำดับเท่ากัน เช่น 1, 1, 3 |
| `dense_rank` | จัดอันดับแบบไม่เว้น gap เมื่อมี tie หรือค่าที่ใช้เรียงลำดับเท่ากัน เช่น 1, 1, 2 |
| `lag` | ดึงค่าจาก row ก่อนหน้าใน partition เดียวกัน โดยอิงลำดับจาก `orderBy` (มองย้อนขึ้นไป 1 row) |
| `lead` | ดึงค่าจาก row ถัดไปใน partition เดียวกัน โดยอิงลำดับจาก `orderBy` (มองลงไปข้างหน้า 1 row) |
| Window frame | ขอบเขต rows ที่ใช้คำนวณ เช่น ตั้งแต่ row แรกจนถึง current row |


## Concept explanation

### Window Function คืออะไร?

`groupBy().agg()` จะรวมหลาย rows ให้กลายเป็น summary row ต่อ group เช่น ยอดขายรวมต่อ customer

แต่ Window Function จะคำนวณภายใน group เหมือนกัน โดยยังคง row-level detail ไว้ เช่น:

```python
row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("updated_at")))
```

ผลลัพธ์คือทุก row ยังอยู่ครบ แต่จะมี column ใหม่ที่บอกว่า row นี้เป็นลำดับที่เท่าไรภายใน `customer_id` นั้น

พูดง่าย ๆ:

> `groupBy` = ยุบ rows เป็น summary  
> `window` = คำนวณข้าม rows แต่ยังไม่ยุบ rows

### `partitionBy` ควรคิดอย่างไร?

`partitionBy` คือ business group ที่เราต้องการให้ window function ทำงานแยกกัน เช่น:

- latest record ต่อ `customer_id`
- top transaction ต่อ `region`
- previous amount ต่อ `customer_id`
- running total ต่อ `customer_id`

ถ้าไม่ใส่ `partitionBy` Spark จะมองทั้ง DataFrame เป็น partition เดียว ซึ่งมักไม่ใช่สิ่งที่ต้องการและอาจแพงมากในข้อมูลจริง

### `orderBy` สำคัญมาก

Window Function หลายตัวต้องพึ่ง order เช่น `row_number`, `lag`, `lead`

ถ้า `orderBy` ไม่ deterministic เช่น มีหลาย records ที่ timestamp เท่ากัน แต่ไม่มี tie-breaker ผลลัพธ์อาจไม่ stable ระหว่าง run ได้

ตัวอย่าง production-friendly ordering:

```python
Window.partitionBy("customer_id").orderBy(
    F.desc("updated_at"),
    F.desc("ingestion_sequence")
)
```

ในตัวอย่างนี้ ถ้า `updated_at` เท่ากัน จะใช้ `ingestion_sequence` เป็น tie-breaker หรือตัวตัดสินลำดับต่อ

### Window Function ใช้แทน join ได้บางกรณี

บางโจทย์ที่ต้องการ previous row หรือ latest row สามารถใช้ `lag`, `lead`, `row_number` ได้โดยไม่ต้อง self-join ซึ่งอ่านง่ายกว่าและลดความเสี่ยงเรื่อง row multiplication จาก join ผิด key


## Mermaid diagram: Window Function Flow

```mermaid
flowchart LR
    A[Input rows] --> B[Partition by business key]
    B --> C[Order rows inside each partition]
    C --> D[Apply window function]
    D --> E[Add result column]
    E --> F[Filter or analyze result]

    B --> B1[Example: customer_id]
    C --> C1[Example: updated_at DESC]
    D --> D1[row_number / rank / lag / lead]
```

Key idea:

> Window Function เริ่มจากเลือก group ด้วย `partitionBy` แล้วจัดลำดับใน group ด้วย `orderBy` จากนั้นค่อยคำนวณ function ที่ต้องการ


## Setup / imports


In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 4, Finished, Available, Finished, False)

## Create mock data

เราจะใช้ 2 DataFrame หลัก:

1. `df_customer_history` สำหรับโจทย์ latest record / deduplication
2. `df_transactions` สำหรับโจทย์ ranking, previous value และ running total


In [4]:
customer_history_data = [
    ("C001", "Alice", "active",   "Bangkok",    "CRM",    "2025-04-01 09:00:00", 1),
    ("C001", "Alice", "inactive", "Bangkok",    "CRM",    "2025-04-03 10:30:00", 2),
    ("C001", "Alice", "active",   "Bangkok",    "CRM",    "2025-04-05 08:20:00", 3),
    ("C002", "Bob",   "active",   "Chiang Mai", "Mobile", "2025-04-01 11:00:00", 1),
    ("C002", "Bob",   "active",   "Chiang Mai", "Mobile", "2025-04-01 11:00:00", 2),
    ("C002", "Bob",   "inactive", "Chiang Mai", "CRM",    "2025-04-07 15:45:00", 3),
    ("C003", "Cara",  "pending",  "Phuket",     "CRM",    "2025-04-02 13:00:00", 1),
    ("C003", "Cara",  "active",   "Phuket",     "CRM",    "2025-04-06 09:30:00", 2),
    ("C004", "Dan",   "active",   "Bangkok",    "Mobile", "2025-04-04 17:10:00", 1),
]

customer_history_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at_raw", T.StringType(), True),
    T.StructField("ingestion_sequence", T.IntegerType(), True),
])

df_customer_history = (
    spark.createDataFrame(customer_history_data, customer_history_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

df_customer_history.show(truncate=False)
df_customer_history.printSchema()

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 6, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|customer_id|customer_name|customer_status|region    |source_system|ingestion_sequence|updated_at         |
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|C001       |Alice        |active         |Bangkok   |CRM          |1                 |2025-04-01 09:00:00|
|C001       |Alice        |inactive       |Bangkok   |CRM          |2                 |2025-04-03 10:30:00|
|C001       |Alice        |active         |Bangkok   |CRM          |3                 |2025-04-05 08:20:00|
|C002       |Bob          |active         |Chiang Mai|Mobile       |1                 |2025-04-01 11:00:00|
|C002       |Bob          |active         |Chiang Mai|Mobile       |2                 |2025-04-01 11:00:00|
|C002       |Bob          |inactive       |Chiang Mai|CRM          |3                 |2025-04-07 15:45:00|
|C003       |Cara         |p

In [5]:
transactions_data = [
    ("T001", "C001", "Bangkok",    "2025-04-01 10:00:00", 1200.50, "success"),
    ("T002", "C001", "Bangkok",    "2025-04-02 11:15:00",  850.00, "success"),
    ("T003", "C001", "Bangkok",    "2025-04-05 14:20:00", 2200.00, "success"),
    ("T004", "C002", "Chiang Mai", "2025-04-01 09:30:00",  450.00, "failed"),
    ("T005", "C002", "Chiang Mai", "2025-04-03 16:00:00", 1800.00, "success"),
    ("T006", "C002", "Chiang Mai", "2025-04-07 12:45:00", 1800.00, "success"),
    ("T007", "C003", "Phuket",     "2025-04-02 08:10:00",  300.00, "success"),
    ("T008", "C003", "Phuket",     "2025-04-06 18:25:00",  950.00, "success"),
    ("T009", "C004", "Bangkok",    "2025-04-04 20:00:00", 1500.00, "success"),
    ("T010", "C004", "Bangkok",    "2025-04-05 21:30:00", 1500.00, "success"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("transaction_ts_raw", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
])

df_transactions = (
    spark.createDataFrame(transactions_data, transaction_schema)
    .withColumn("transaction_ts", F.to_timestamp("transaction_ts_raw"))
    .drop("transaction_ts_raw")
)

df_transactions.show(truncate=False)
df_transactions.printSchema()

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 7, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |
+--------------+-----------+----------+------+-------+-------------------+
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|
|T004          |C002       |Chiang Mai|450.0 |failed |2025-04-01 09:30:00|
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|
|T007          |C003       |Phuket    |300.0 |success|2025-04-02 08:10:00|
|T008          |C003       |Phuket    |950.0 |success|2025-04-06 18:25:00|
|T009          |C004       |Bangkok   |1500.0|success|2025-04-04 20:00:00|
|T010          |C004       |Bangkok   |1500.0|success|2025-04-05 21:30:00|
+--------------+---------

## PySpark code examples

Section นี้จะค่อย ๆ ใช้ Window Functions จาก case ที่ง่ายไปหายากขึ้น:

1. สร้าง Window specification
2. เลือก latest record ด้วย `row_number`
3. จัดอันดับด้วย `rank` และ `dense_rank`
4. เปรียบเทียบ previous value ด้วย `lag`
5. คำนวณ running total ด้วย window frame


### Example 1: Define a window specification

เริ่มจากกำหนด window ว่าเราจะ partition ด้วย `customer_id` และเรียงลำดับจาก `updated_at` ล่าสุดก่อน

เพิ่ม `ingestion_sequence` เป็น tie-breaker เพื่อให้ผลลัพธ์ deterministic กรณี timestamp เท่ากัน


In [6]:
latest_customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.desc("updated_at"), F.desc("ingestion_sequence"))
)

print("Window specification has been defined.")

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 8, Finished, Available, Finished, False)

Window specification has been defined.


### Example 2: Select latest customer record using `row_number`

`row_number()` จะให้เลขลำดับภายในแต่ละ `customer_id`

ถ้า filter `row_number = 1` จะได้ record ล่าสุดของแต่ละ customer ตาม order ที่กำหนด


In [7]:
df_customer_with_row_number = (
    df_customer_history
    .withColumn("row_number", F.row_number().over(latest_customer_window))
    .orderBy("customer_id", "row_number")
)

df_customer_with_row_number.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 9, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+-------------+------------------+-------------------+----------+
|customer_id|customer_name|customer_status|region    |source_system|ingestion_sequence|updated_at         |row_number|
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+----------+
|C001       |Alice        |active         |Bangkok   |CRM          |3                 |2025-04-05 08:20:00|1         |
|C001       |Alice        |inactive       |Bangkok   |CRM          |2                 |2025-04-03 10:30:00|2         |
|C001       |Alice        |active         |Bangkok   |CRM          |1                 |2025-04-01 09:00:00|3         |
|C002       |Bob          |inactive       |Chiang Mai|CRM          |3                 |2025-04-07 15:45:00|1         |
|C002       |Bob          |active         |Chiang Mai|Mobile       |2                 |2025-04-01 11:00:00|2         |
|C002       |Bob          |active         |Chian

In [8]:
df_latest_customer = (
    df_customer_with_row_number
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .orderBy("customer_id")
)

df_latest_customer.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 10, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|customer_id|customer_name|customer_status|region    |source_system|ingestion_sequence|updated_at         |
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|C001       |Alice        |active         |Bangkok   |CRM          |3                 |2025-04-05 08:20:00|
|C002       |Bob          |inactive       |Chiang Mai|CRM          |3                 |2025-04-07 15:45:00|
|C003       |Cara         |active         |Phuket    |CRM          |2                 |2025-04-06 09:30:00|
|C004       |Dan          |active         |Bangkok   |Mobile       |1                 |2025-04-04 17:10:00|
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+



### Example 3: Use `rank` and `dense_rank` for transaction ranking

`rank()` และ `dense_rank()` เหมาะกับการจัดอันดับภายใน group เช่น top transaction ต่อ region

ความต่างสำคัญ:

- `rank`: ถ้ามี tie หรือค่าที่ใช้เรียงลำดับเท่ากัน จะเว้นอันดับ เช่น 1, 1, 3
- `dense_rank`: ถ้ามี tie หรือค่าที่ใช้เรียงลำดับเท่ากัน จะไม่เว้นอันดับ เช่น 1, 1, 2

Use Cases:

- ใช้ `rank()` เมื่ออันดับแบบมี gap มีความหมาย เช่น leaderboard หรือ sales ranking  
- ใช้ `dense_rank()` เมื่ออยากจัดอันดับแบบต่อเนื่อง เช่น top N groups หรือ tier ranking

In [9]:
region_amount_window = (
    Window
    .partitionBy("region")
    .orderBy(F.desc("amount"))
)

df_transaction_ranking = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("rank_in_region", F.rank().over(region_amount_window))
    .withColumn("dense_rank_in_region", F.dense_rank().over(region_amount_window))
    .orderBy("region", "rank_in_region", "transaction_id")
)

df_transaction_ranking.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 11, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+--------------+--------------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |rank_in_region|dense_rank_in_region|
+--------------+-----------+----------+------+-------+-------------------+--------------+--------------------+
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|1             |1                   |
|T009          |C004       |Bangkok   |1500.0|success|2025-04-04 20:00:00|2             |2                   |
|T010          |C004       |Bangkok   |1500.0|success|2025-04-05 21:30:00|2             |2                   |
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|4             |3                   |
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|5             |4                   |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|1             |1                   |
|

### Example 4: Compare current row with previous row using `lag`

`lag()` ใช้ดูค่าจาก row ก่อนหน้าใน partition เดียวกัน เช่น transaction ก่อนหน้าของ customer คนเดียวกัน

ใน case นี้เราจะหา:

- `previous_amount`
- `amount_change`
- `previous_transaction_ts`


In [10]:
customer_transaction_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_ts", "transaction_id")
)

df_transaction_with_previous = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("previous_amount", F.lag("amount").over(customer_transaction_window))
    .withColumn("previous_transaction_ts", F.lag("transaction_ts").over(customer_transaction_window))
    .withColumn("amount_change", F.round(F.col("amount") - F.col("previous_amount"), 2))
    .orderBy("customer_id", "transaction_ts")
)

df_transaction_with_previous.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 12, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+---------------+-----------------------+-------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |previous_amount|previous_transaction_ts|amount_change|
+--------------+-----------+----------+------+-------+-------------------+---------------+-----------------------+-------------+
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|NULL           |NULL                   |NULL         |
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|1200.5         |2025-04-01 10:00:00    |-350.5       |
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|850.0          |2025-04-02 11:15:00    |1350.0       |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|NULL           |NULL                   |NULL         |
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|1800.0         |2025-04

### Example 5: Look ahead using `lead`

`lead()` คือ opposite ของ `lag()` ใช้ดูค่าจาก row ถัดไปใน partition เดียวกัน

ตัวอย่างนี้ช่วยตอบคำถามว่า transaction ถัดไปของ customer คือเมื่อไร


In [11]:
df_transaction_with_next = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("next_transaction_ts", F.lead("transaction_ts").over(customer_transaction_window))
    .withColumn("next_amount", F.lead("amount").over(customer_transaction_window))
    .orderBy("customer_id", "transaction_ts")
)

df_transaction_with_next.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 13, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+-------------------+-----------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |next_transaction_ts|next_amount|
+--------------+-----------+----------+------+-------+-------------------+-------------------+-----------+
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|2025-04-02 11:15:00|850.0      |
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|2025-04-05 14:20:00|2200.0     |
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|NULL               |NULL       |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|2025-04-07 12:45:00|1800.0     |
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|NULL               |NULL       |
|T007          |C003       |Phuket    |300.0 |success|2025-04-02 08:10:00|2025-04-06 18:25:00|950.0      |
|T008          |C003       |Phuket   

### Example 6: Running total using window frame

Window frame กำหนดช่วง rows ที่จะใช้คำนวณ

ตัวอย่างนี้ใช้:

```python
rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

หมายถึง เริ่มตั้งแต่ row แรกของ customer นั้นจนถึง current row เพื่อคำนวณ cumulative amount


In [12]:
running_total_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_ts", "transaction_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_transaction_running_total = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("running_success_amount", F.round(F.sum("amount").over(running_total_window), 2))
    .orderBy("customer_id", "transaction_ts")
)

df_transaction_running_total.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 14, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+----------------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |running_success_amount|
+--------------+-----------+----------+------+-------+-------------------+----------------------+
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|1200.5                |
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|2050.5                |
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|4250.5                |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|1800.0                |
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|3600.0                |
|T007          |C003       |Phuket    |300.0 |success|2025-04-02 08:10:00|300.0                 |
|T008          |C003       |Phuket    |950.0 |success|2025-04-06 18:25:00|1250.0                |
|T009          |C004

### Example 7: Window Function is still lazy until action

Window Function เป็น transformation เหมือน `select`, `filter`, `withColumn`

Spark จะยังไม่ execute จริงจนกว่าจะเจอ action เช่น `.show()`, `.count()`, `.write`


In [13]:
df_latest_success_transaction = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("customer_id")
            .orderBy(F.desc("transaction_ts"), F.desc("transaction_id"))
        )
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Latest transaction logic has been defined, but no action has run yet.")

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 15, Finished, Available, Finished, False)

Latest transaction logic has been defined, but no action has run yet.


In [14]:
df_latest_success_transaction.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 16, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |
+--------------+-----------+----------+------+-------+-------------------+
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|
|T008          |C003       |Phuket    |950.0 |success|2025-04-06 18:25:00|
|T010          |C004       |Bangkok   |1500.0|success|2025-04-05 21:30:00|
+--------------+-----------+----------+------+-------+-------------------+



## Quick recap

| Question | Answer |
|---|---|
| Window Function ใช้ทำอะไร? | คำนวณข้าม rows ภายใน group โดยยังคง row-level detail ไว้ |
| `partitionBy` คืออะไร? | กำหนด group ที่ window function จะทำงานแยกกัน |
| `orderBy` สำคัญอย่างไร? | กำหนดลำดับภายใน partition โดยเฉพาะกับ `row_number`, `lag`, `lead` |
| `row_number()` ใช้บ่อยกับอะไร? | latest-record selection และ deduplication ตาม business key + timestamp |
| `rank()` vs `dense_rank()` ต่างกันอย่างไร? | `rank` เว้นอันดับเมื่อมี tie หรือค่าที่ใช้เรียงลำดับเท่ากัน แต่ `dense_rank` ไม่เว้นอันดับ |
| `lag()` ใช้ทำอะไร? | ดึงค่าจาก previous row ใน partition เดียวกัน |
| Window Function execute ทันทีไหม? | ไม่ทันที ยังเป็น transformation และจะ execute เมื่อเจอ action |


## Exercises

### Exercise 1: Create latest customer snapshot

สร้าง DataFrame ชื่อ `df_ex_latest_customer_snapshot` ที่เก็บ customer record ล่าสุดต่อ `customer_id`

Requirements:

- ใช้ `row_number()`
- partition ด้วย `customer_id`
- order ด้วย `updated_at` ล่าสุดก่อน และ `ingestion_sequence` ล่าสุดก่อน
- เหลือ 1 row ต่อ customer
- แสดงผลลัพธ์ด้วย `.show()`

Expected result:

- `C001` ควรมี status ล่าสุดเป็น `active`
- `C002` ควรมี status ล่าสุดเป็น `inactive`
- จำนวน rows ควรเท่ากับจำนวน customer_id ที่ไม่ซ้ำ


In [15]:
exercise_latest_customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.desc("updated_at"), F.desc("ingestion_sequence"))
)

df_ex_latest_customer_snapshot = (
    df_customer_history
    .withColumn("row_number", F.row_number().over(exercise_latest_customer_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .orderBy("customer_id")
)

df_ex_latest_customer_snapshot.show(truncate=False)

print("Latest customer snapshot count:", df_ex_latest_customer_snapshot.count())

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 17, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|customer_id|customer_name|customer_status|region    |source_system|ingestion_sequence|updated_at         |
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+
|C001       |Alice        |active         |Bangkok   |CRM          |3                 |2025-04-05 08:20:00|
|C002       |Bob          |inactive       |Chiang Mai|CRM          |3                 |2025-04-07 15:45:00|
|C003       |Cara         |active         |Phuket    |CRM          |2                 |2025-04-06 09:30:00|
|C004       |Dan          |active         |Bangkok   |Mobile       |1                 |2025-04-04 17:10:00|
+-----------+-------------+---------------+----------+-------------+------------------+-------------------+

Latest customer snapshot count: 4


### Exercise 2: Find top 2 successful transactions per region

สร้าง DataFrame ชื่อ `df_ex_top2_region_transactions` เพื่อหา top 2 successful transactions ต่อ `region`

Requirements:

- ใช้เฉพาะ records ที่ `status = 'success'`
- partition ด้วย `region`
- order ด้วย `amount` จากมากไปน้อย
- ใช้ `dense_rank()` เพื่อให้ tie amount ได้อันดับเดียวกัน
- filter เฉพาะ `dense_rank_in_region <= 2`
- แสดงผลลัพธ์ด้วย `.show()`

Expected result:

- ในแต่ละ region ถ้ามี records ที่ `transaction_amount` เท่ากัน ควรได้ `dense_rank` อันดับเดียวกัน และอันดับถัดไปไม่เว้นเลข
- ผลลัพธ์ควรเป็น top records ต่อ region ไม่ใช่ top records ทั้ง DataFrame


In [16]:
exercise_region_window = (
    Window
    .partitionBy("region")
    .orderBy(F.desc("amount"))
)

df_ex_top2_region_transactions = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("dense_rank_in_region", F.dense_rank().over(exercise_region_window))
    .filter(F.col("dense_rank_in_region") <= 2)
    .orderBy("region", "dense_rank_in_region", "transaction_id")
)

df_ex_top2_region_transactions.show(truncate=False)

StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 18, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+--------------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |dense_rank_in_region|
+--------------+-----------+----------+------+-------+-------------------+--------------------+
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|1                   |
|T009          |C004       |Bangkok   |1500.0|success|2025-04-04 20:00:00|2                   |
|T010          |C004       |Bangkok   |1500.0|success|2025-04-05 21:30:00|2                   |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|1                   |
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|1                   |
|T008          |C003       |Phuket    |950.0 |success|2025-04-06 18:25:00|1                   |
|T007          |C003       |Phuket    |300.0 |success|2025-04-02 08:10:00|2                   |
+--------------+-----------+----------+-

### Exercise 3: Detect amount increase from previous successful transaction

สร้าง DataFrame ชื่อ `df_ex_amount_trend` เพื่อเปรียบเทียบ amount ปัจจุบันกับ successful transaction ก่อนหน้าของ customer คนเดียวกัน

Requirements:

- ใช้เฉพาะ records ที่ `status = 'success'`
- partition ด้วย `customer_id`
- order ด้วย `transaction_ts`, `transaction_id`
- ใช้ `lag("amount")` เพื่อสร้าง `previous_amount`
- สร้าง column `amount_trend` โดยมีค่า:
  - `first_transaction` ถ้ายังไม่มี previous amount
  - `increase` ถ้า amount มากกว่า previous amount
  - `decrease` ถ้า amount น้อยกว่า previous amount
  - `same` ถ้า amount เท่ากับ previous amount
- แสดงผลลัพธ์ด้วย `.show()`

Expected result:

- transaction แรกของแต่ละ customer ควรเป็น `first_transaction`
- ถ้า amount เท่ากันกับ previous amount ควรเป็น `same`


In [17]:
exercise_customer_transaction_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_ts", "transaction_id")
)

df_ex_amount_trend = (
    df_transactions
    .filter(F.col("status") == "success")
    .withColumn("previous_amount", F.lag("amount").over(exercise_customer_transaction_window))
    .withColumn(
        "amount_trend",
        F.when(F.col("previous_amount").isNull(), F.lit("first_transaction"))
         .when(F.col("amount") > F.col("previous_amount"), F.lit("increase"))
         .when(F.col("amount") < F.col("previous_amount"), F.lit("decrease"))
         .otherwise(F.lit("same"))
    )
    .orderBy("customer_id", "transaction_ts")
)

df_ex_amount_trend.show(truncate=False)


StatementMeta(, 93ba9425-0357-42fd-83be-7deb8d2d8513, 19, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+-------------------+---------------+-----------------+
|transaction_id|customer_id|region    |amount|status |transaction_ts     |previous_amount|amount_trend     |
+--------------+-----------+----------+------+-------+-------------------+---------------+-----------------+
|T001          |C001       |Bangkok   |1200.5|success|2025-04-01 10:00:00|NULL           |first_transaction|
|T002          |C001       |Bangkok   |850.0 |success|2025-04-02 11:15:00|1200.5         |decrease         |
|T003          |C001       |Bangkok   |2200.0|success|2025-04-05 14:20:00|850.0          |increase         |
|T005          |C002       |Chiang Mai|1800.0|success|2025-04-03 16:00:00|NULL           |first_transaction|
|T006          |C002       |Chiang Mai|1800.0|success|2025-04-07 12:45:00|1800.0         |same             |
|T007          |C003       |Phuket    |300.0 |success|2025-04-02 08:10:00|NULL           |first_transaction|
|T008          |C00

## Common mistakes

1. **ลืมใส่ `partitionBy`**

   ถ้าไม่ใส่ `partitionBy` Spark จะคำนวณ window บนทั้ง DataFrame เป็นกลุ่มเดียว ซึ่งมักผิด business logic และแพงมากเมื่อข้อมูลใหญ่

2. **ใช้ `orderBy` ที่ไม่ deterministic**

   ถ้า timestamp เท่ากันหลาย records แต่ไม่มี tie-breaker เช่น `ingestion_sequence` หรือ `transaction_id` ผลลัพธ์จาก `row_number()` อาจไม่ stable

3. **ใช้ `rank()` แทน `row_number()` ใน latest-record deduplication**

   ถ้ามี tie, `rank()` อาจให้หลาย rows เป็นอันดับ 1 ทำให้ filter แล้วเหลือมากกว่า 1 record ต่อ key ได้ ถ้าต้องการเหลือ record เดียว มักใช้ `row_number()` พร้อม tie-breaker

4. **คิดว่า Window Function จะลดจำนวน rows เหมือน `groupBy`**

   Window Function เพิ่ม column คำนวณ แต่จำนวน rows ยังเท่าเดิม จนกว่าเราจะ filter เอง เช่น filter `row_number = 1`

5. **ใช้ window กว้างเกินจำเป็น**

   Window ที่ไม่มี `partitionBy` หรือมีบางกลุ่มข้อมูลใหญ่มาก เช่น key เดียวมี rows จำนวนมาก อาจทำให้ shuffle และ sort หนัก ควรเลือก `partitionBy` key ที่สอดคล้องกับ business logic และตรวจ plan เมื่อ performance สำคัญ

   - `partitionBy` ใน Window Function คือ logical group ของข้อมูล ไม่ใช่จำนวน physical Spark partitions โดยตรง
   - **Shuffle:** Spark อาจต้องย้ายข้อมูลตาม key เพื่อให้ records ที่อยู่ใน group เดียวกันมาคำนวณร่วมกัน
   - **Sort:** Spark ต้องเรียง rows ภายในแต่ละ group ตาม `orderBy` ถ้า group ใหญ่มากอาจใช้ memory สูงหรือ job ช้าลง

6. **drop helper columns เร็วเกินไปตอน debug**

   Columns เช่น `row_number`, `previous_amount`, `dense_rank` ช่วยตรวจ logic ได้ ก่อน drop ควรดูผลลัพธ์ให้มั่นใจก่อน


## Expected Output / Success Criteria

เมื่อจบ Day 07 ควรทำได้:

- อธิบายได้ว่า Window Function คำนวณข้าม rows ภายใน group โดยยังคง row-level detail ไว้
- แยกได้ว่า `groupBy().agg()` ยุบ rows แต่ Window Function ไม่ยุบ rows โดยอัตโนมัติ
- ใช้ `Window.partitionBy()` เพื่อกำหนด business group ได้
- ใช้ `Window.orderBy()` เพื่อกำหนดลำดับภายใน partition ได้
- ใช้ `row_number()` เพื่อเลือก latest record ต่อ key ได้
- ใช้ tie-breaker เช่น `ingestion_sequence` หรือ `transaction_id` เพื่อทำให้ผลลัพธ์ deterministic ได้
- ใช้ `rank()` และ `dense_rank()` เพื่อจัดอันดับ records ภายใน group ได้
- ใช้ `lag()` และ `lead()` เพื่อดู previous / next row ได้
- ใช้ window frame เพื่อคำนวณ running total ได้
- อธิบาย production risk ของ window function ได้ เช่น wrong partition, unstable ordering, large shuffle และ helper column debug


## Final takeaway

Window Functions เป็นหนึ่งในเครื่องมือสำคัญของ PySpark สำหรับโจทย์ที่ต้องคิดแบบ “ภายใน group มีลำดับของ records”

> ก่อนใช้ Window Function ให้ถามตัวเองเสมอว่า partition คืออะไร และ order ที่ถูกต้องคืออะไร

Mindset ที่สำคัญสำหรับ Data Engineer คือ:

- ก่อนเลือก latest record ต้องรู้ business key และ timestamp ที่เชื่อถือได้
- ถ้า timestamp ซ้ำ ต้องมี tie-breaker
- ถ้าต้องการเหลือ 1 record ต่อ key ใช้ `row_number()` มากกว่า `rank()`
- ถ้าต้องดู previous / next event ภายในกลุ่มเดียวกัน เช่น ต่อ `customer_id` หรือ `region` ให้พิจารณาใช้ `lag()` / `lead()` แทน self-join
- หลังใช้ window ควรตรวจ output ด้วย `.show()`, `.count()` หรือ `groupBy(...).count()` เพื่อเช็กว่าจำนวน rows ต่อ group ถูกต้องก่อนนำไป write ต่อ